# Jupyter notebook tutorial with three types of visualizations!

In [1]:
### Original ata source --> https://gis-txdot.opendata.arcgis.com/datasets/TXDOT::txdot-roadway-inventory/about

In [2]:
import pandas as pd
import geopandas as gpd ## my personal favorite python package <<<3333 plz respect the goat geopandas (and the amazing packages it is built on top of, i.e. leaflet)

import h3pandas
import pydeck

## download pre-loaded geopackage. Note that this data has been altered from its original source by me.

In [3]:
houston_area_road_inventory = gpd.read_file("Houston_Area_road_VMT.gpkg")

In [19]:
## prepare to make centroid and lat / long frames

houston_area_road_inventory['geometry'] = houston_area_road_inventory['geometry'].centroid
houston_area_road_inventory['lat'] = houston_area_road_inventory['geometry'].y
houston_area_road_inventory['long'] = houston_area_road_inventory['geometry'].x

C:\Users\aaron\AppData\Local\Temp\ipykernel_6076\2793356874.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  houston_area_road_inventory['geometry'] = houston_area_road_inventory['geometry'].centroid


In [5]:
## making two different frames here to reduce the data load that each map carries.

pydeck_data = houston_area_road_inventory[['DVMT', 'lat', 'long']] ## this one is to make the PyDeck 'hexagonlayer'
centroid_frame = houston_area_road_inventory[['DVMT', 'geometry']] ## this one is for the H3 visualization

## H3 Visualization

In [6]:
hexagon_frame = centroid_frame.h3.geo_to_h3_aggregate(8) ## invoking H3 in one simple line!

In [7]:
#hexMap = hexagon_frame.explore(column = 'DVMT', vmax = 100000) ## declare the map

In [8]:
#hexMap

## Pydeck Hexagon Visualization

In [9]:
## check out different PyDeck options here ---> https://deckgl.readthedocs.io/en/latest/#gallery

In [10]:
layer = pydeck.Layer(
    "HexagonLayer",
    pydeck_data,
    get_position=["long", "lat"],
    radius = 1000, ## changes the size of the hexes, important!

    get_elevation_weight = 'DVMT', ##determines the height
    get_color_weight = 'DVMT', ##determines the color

    opacity = 0.4,
    auto_highlight=True,
    elevation_scale=50,
    pickable=True,
    elevation_range=[0, 3000],
    extruded=True,
    coverage=1,
)

In [11]:
## determines the starting view 
view_state = pydeck.ViewState(
    longitude=-95.415,
    latitude=29.6,
    zoom=8,
    min_zoom=5,
    max_zoom=15,
    pitch=0, ## adjusts the viewing height
    bearing=0, ## adjusts where the 'camera' is looking on from, in the E/W direction
)

In [12]:
#pdk_hex_map_render = pydeck.Deck(layers=[layer], initial_view_state=view_state) ## declares the map

In [13]:
#pdk_hex_map_render

## Pydeck Heatmap Layer

In [14]:
COLOR_BREWER_BLUE_SCALE = [
    [240, 249, 232],
    [204, 235, 197],
    [168, 221, 181],
    [123, 204, 196],
    [67, 162, 202],
    [8, 104, 172],
]

In [15]:
vmt_heat = pydeck.Layer(
    "HeatmapLayer",
    data=pydeck_data,
    opacity=0.9,
    get_position=["long", "lat"],
    #aggregation=pydeck.types.String("MEAN"),
    color_range=COLOR_BREWER_BLUE_SCALE,
    threshold=1,
    get_weight="DVMT",
    pickable=True,
)

In [16]:
heatmap_render = pydeck.Deck(
    layers=[vmt_heat], ## can easily include multiple layers here
    initial_view_state=view_state,
    tooltip={"text": "Daily Vehicle Miles Traveled"},
)

In [18]:
#heatmap_render